In [ ]:
import pandas as pd
import io
from collatex import Collation, collate
from nltk.stem import WordNetLemmatizer
import nltk
import warnings
warnings.filterwarnings('ignore')

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
lemmatizer = WordNetLemmatizer()

df = pd.read_csv('clean_duo_data.csv')  

story_counts = df.groupby('STORY')['PUBLISHER'].nunique()
valid_stories = story_counts[story_counts >= 2].index
target_story = valid_stories[0]
story_df = df[df['STORY'] == target_story].reset_index(drop=True)

print(f" Aligning Story: {target_story}")
print(f" Sources: {story_df['PUBLISHER'].unique()}")

collation = Collation()
for i, row in story_df.iterrows():
    title = str(row['TITLE']).strip()
    pub = row['PUBLISHER']
    if len(title) < 3: continue
    collation.add_plain_witness(f"{pub}_{i}", title)

print(" Aligning headlines...")
csv_str = collate(collation, output='csv')
alignment = pd.read_csv(io.StringIO(csv_str), index_col=0)

def categorize_position(row):
    tokens = [str(t).strip() for t in row if pd.notna(t) and str(t).strip() != '']
    if len(tokens) <= 1: return 'same'
    
    unique_tokens = list(set(tokens))
    lemmatized = [lemmatizer.lemmatize(t.lower()) for t in unique_tokens]
    
    if len(set(lemmatized)) == 1:
        return 'morphological'
    elif any(t.startswith(('+', '-')) or t.replace('.','',1).isdigit() for t in tokens):
        return 'plus/minus'
    else:
        return 'lexical'

variant_counts = {'same': 0, 'lexical': 0, 'morphological': 0, 'plus/minus': 0}
for _, row in alignment.iterrows():
    variant_counts[categorize_position(row)] += 1

print("\n Variant Distribution:")
for cat, count in variant_counts.items():
    print(f"   {cat.upper():<15} | {count}")

print("\n Sample Alignment Table (first 6 positions):")
print(alignment.iloc[:, :6])

🔍 Aligning Story: d--wbowLE_mEcaMG2HSlJBQmHDBVM
📰 Sources: ['Reuters' 'MarketWatch']
⚡ Aligning headlines...

📊 Variant Distribution:
   SAME            | 0
   LEXICAL         | 2
   MORPHOLOGICAL   | 0
   PLUS/MINUS      | 3

📝 Sample Alignment Table (first 6 positions):
               UPDATE 2-  China                    c.            Unnamed: 4  \
Reuters_0                                                                     
MarketWatch_1        NaN  China   loans slow in April                   NaN   
Reuters_2            NaN  China                April                    NaN   
Reuters_3            NaN  China                April                   new    
Reuters_4            NaN  China                April   fiscal revenues up 9   
Reuters_5            NaN  China                April   fiscal revenues up 9   

               bank  tells banks to quicken mortgage lending  
Reuters_0                                                     
MarketWatch_1    NaN                           

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from collatex import Collation, collate
import io
import warnings
warnings.filterwarnings('ignore')

def extract_variant_features(df):
    """Extract story-level variant statistics using CollateX"""
    story_features = []
    
    for story_id in df['STORY'].unique():
        story_df = df[df['STORY'] == story_id].reset_index(drop=True)
        
        if len(story_df) < 2:
            continue
            
        collation = Collation()
        for i, row in story_df.iterrows():
            title = str(row['TITLE']).strip()
            pub = row['PUBLISHER']
            if len(title) < 3: continue
            collation.add_plain_witness(f"{pub}_{i}", title)
        
        try:
            csv_str = collate(collation, output='csv')
            alignment = pd.read_csv(io.StringIO(csv_str), index_col=0)
            
            lexical_count = 0
            plus_minus_count = 0
            total_positions = alignment.shape[1]
            
            for col in alignment.columns:
                tokens = alignment[col].dropna().astype(str).str.strip().tolist()
                tokens = [t for t in tokens if t != '']
                
                if len(tokens) <= 1:
                    continue
                    
                if any(t.replace('+', '').replace('-', '').replace('.', '').isdigit() for t in tokens):
                    plus_minus_count += 1
                else:
                    lexical_count += 1
            
            story_features.append({
                'STORY': story_id,
                'lexical_ratio': lexical_count / max(total_positions, 1),
                'plus_minus_ratio': plus_minus_count / max(total_positions, 1),
                'total_variants': (lexical_count + plus_minus_count) / max(total_positions, 1)
            })
            
        except Exception as e:
            story_features.append({
                'STORY': story_id,
                'lexical_ratio': 0.0,
                'plus_minus_ratio': 0.0,
                'total_variants': 0.0
            })
    
    return pd.DataFrame(story_features)

def run_variant_aware_experiment(filepath, dataset_name):
    print(f"\n{'='*60}")
    print(f" APRIL: VARIANT-AWARE MODEL - {dataset_name.upper()}")
    print(f"{'='*60}")
    
    df = pd.read_csv(filepath)
    print(f" Loaded {len(df)} articles from {df['PUBLISHER'].nunique()} sources")
    
    print(" Extracting variant-based features...")
    variant_feats = extract_variant_features(df)
    
    df_merged = df.merge(variant_feats, on='STORY', how='left')
    df_merged.fillna(0, inplace=True)
    
    X_text = df_merged['TITLE']
    X_variant = df_merged[['lexical_ratio', 'plus_minus_ratio', 'total_variants']].values
    y = df_merged['PUBLISHER']
    
    tfidf = TfidfVectorizer(analyzer='char', ngram_range=(2, 4), max_features=5000)
    X_tfidf = tfidf.fit_transform(X_text)
    
    from scipy.sparse import hstack
    X_combined = hstack([X_tfidf, X_variant])
    
    model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    acc_scores = cross_val_score(model, X_combined, y, cv=cv, scoring='accuracy')
    f1_scores = cross_val_score(model, X_combined, y, cv=cv, scoring='f1_macro')
    
    print(f"\n Results:")
    print(f"   Accuracy: {acc_scores.mean():.4f} ± {acc_scores.std():.4f}")
    print(f"   F1-Macro: {f1_scores.mean():.4f} ± {f1_scores.std():.4f}")
    
    return acc_scores.mean(), f1_scores.mean()

duo_acc, duo_f1 = run_variant_aware_experiment('clean_duo_data.csv', 'DUO')
trio_acc, trio_f1 = run_variant_aware_experiment('clean_trio_data.csv', 'TRIO')

print("\n" + "="*50)
print(" FINAL COMPARISON vs. FEBRUARY BASELINE")
print("="*50)
print(f"DUO  | Feb Baseline: 0.890 Acc / 0.882 F1 | April Variant Model: {duo_acc:.3f} Acc / {duo_f1:.3f} F1")
print(f"TRIO | Feb Baseline: 0.803 Acc / 0.788 F1 | April Variant Model: {trio_acc:.3f} Acc / {trio_f1:.3f} F1")


🔬 APRIL: VARIANT-AWARE MODEL - DUO
📂 Loaded 3073 articles from 2 sources
📊 Extracting variant-based features...

✅ Results:
   Accuracy: 0.8854 ± 0.0060
   F1-Macro: 0.8781 ± 0.0060

🔬 APRIL: VARIANT-AWARE MODEL - TRIO
📂 Loaded 2946 articles from 3 sources
📊 Extracting variant-based features...

✅ Results:
   Accuracy: 0.7970 ± 0.0090
   F1-Macro: 0.7831 ± 0.0076

🏆 FINAL COMPARISON vs. FEBRUARY BASELINE
DUO  | Feb Baseline: 0.890 Acc / 0.882 F1 | April Variant Model: 0.885 Acc / 0.878 F1
TRIO | Feb Baseline: 0.803 Acc / 0.788 F1 | April Variant Model: 0.797 Acc / 0.783 F1


In [ ]:
import pandas as pd
import numpy as np
import io
import gc
from collatex import Collation, collate
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sentence_transformers import SentenceTransformer
from gensim.models import Word2Vec, FastText
import warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt', quiet=True)
lemmatizer = WordNetLemmatizer()


def get_collatex_story_features(df):
    """Aligns parallel headlines per STORY and returns variant ratios"""
    story_feats = []
    for story_id, group in df.groupby('STORY'):
        if len(group) < 2: continue
        collation = Collation()
        for i, (_, row) in enumerate(group.iterrows()):
            t = str(row['TITLE']).strip()
            if len(t) < 3: continue
            collation.add_plain_witness(f"w{i}", t)
        try:
            csv_str = collate(collation, output='csv')
            align = pd.read_csv(io.StringIO(csv_str), index_col=0)
            counts = {'same':0, 'lexical':0, 'morphological':0, 'plus/minus':0}
            for col in align.columns:
                toks = [str(v).strip() for v in align[col] if pd.notna(v) and str(v).strip()!='']
                if len(toks) <= 1: 
                    counts['same'] += 1
                    continue
                unique = list(set(toks))
                lemmas = [lemmatizer.lemmatize(w.lower()) for w in unique]
                if len(set(lemmas)) == 1:
                    counts['morphological'] += 1
                elif any(t.startswith(('+','-')) or t.replace('.','',1).isdigit() for t in toks):
                    counts['plus/minus'] += 1
                else:
                    counts['lexical'] += 1
            total = sum(counts.values())
            if total > 0:
                ratios = {k: v/total for k,v in counts.items()}
                story_feats.append({'STORY': story_id, **ratios})
        except: pass
    return pd.DataFrame(story_feats) if story_feats else pd.DataFrame(columns=['STORY','same','lexical','morphological','plus/minus'])


def get_tfidf(texts, analyzer='word', ngram_range=(1,2), max_feat=2500):
    vec = TfidfVectorizer(analyzer=analyzer, ngram_range=ngram_range, max_features=max_feat, sublinear_tf=True)
    return vec.fit_transform(texts).toarray() 

def get_word2vec(texts, dim=100):
    tokenized = [word_tokenize(t.lower()) for t in texts]
    model = Word2Vec(tokenized, vector_size=dim, window=5, min_count=2, workers=4, epochs=10)
    return np.array([np.mean([model.wv[w] for w in tok if w in model.wv], axis=0) if any(w in model.wv for w in tok) else np.zeros(dim) for tok in tokenized])

def get_fasttext(texts, dim=100):
    tokenized = [word_tokenize(t.lower()) for t in texts]
    model = FastText(tokenized, vector_size=dim, window=5, min_count=2, workers=4, epochs=10)
    return np.array([np.mean([model.wv[w] for w in tok if w in model.wv], axis=0) if any(w in model.wv for w in tok) else np.zeros(dim) for tok in tokenized])

def get_sbert(texts):
    model = SentenceTransformer('all-MiniLM-L6-v2')
    return model.encode(texts, show_progress_bar=False, batch_size=32).astype(np.float32)


def run_collatex_enhanced_experiment(filepath, dataset_name, feb_baseline_f1):
    print(f"\n{'='*70}")
    print(f" COLLATEX-ENHANCED PIPELINE - {dataset_name.upper()}")
    print(f"{'='*70}")
    
    df = pd.read_csv(filepath)
    print(" Extracting CollateX variant features per story...")
    collatex_df = get_collatex_story_features(df)
    df = df.merge(collatex_df, on='STORY', how='left').fillna(0)
    X_collatex = df[['same', 'lexical', 'morphological', 'plus/minus']].values
    
    texts = df['TITLE'].values
    labels = df['PUBLISHER'].values
    
    print(" Computing text feature sets...")
    X_tfidf_w = get_tfidf(texts, analyzer='word', ngram_range=(1,2))
    X_tfidf_c = get_tfidf(texts, analyzer='char', ngram_range=(2,4))
    X_w2v = get_word2vec(texts)
    X_ft = get_fasttext(texts)
    X_sbert = get_sbert(texts)
    
    feature_sets = {
        'TF-IDF Word + CollateX': np.hstack([X_tfidf_w, X_collatex]),
        'TF-IDF Char + CollateX': np.hstack([X_tfidf_c, X_collatex]),
        'Word2Vec + CollateX': np.hstack([X_w2v, X_collatex]),
        'FastText + CollateX': np.hstack([X_ft, X_collatex]),
        'SBERT + CollateX': np.hstack([X_sbert, X_collatex]),
        'TF-IDF Word + W2V + CollateX': np.hstack([X_tfidf_w, X_w2v, X_collatex]),
        'TF-IDF Word + FT + CollateX': np.hstack([X_tfidf_w, X_ft, X_collatex]),
        'TF-IDF Word + SBERT + CollateX': np.hstack([X_tfidf_w, X_sbert, X_collatex])
    }
    
    models = {
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
        'Naive Bayes': GaussianNB(),
        'SVM': LinearSVC(random_state=42, max_iter=2000, class_weight='balanced'),
        'KNN': KNeighborsClassifier(n_neighbors=5),
        'SGD': SGDClassifier(loss='log_loss', random_state=42, max_iter=1000, class_weight='balanced')
    }
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    results = []
    
    print(" Training with 5-Fold CV...\n")
    for feat_name, X_feat in feature_sets.items():
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_feat)
        
        for model_name, model in models.items():
            try:
                acc = cross_val_score(model, X_scaled, labels, cv=cv, scoring='accuracy').mean()
                f1 = cross_val_score(model, X_scaled, labels, cv=cv, scoring='f1_macro').mean()
                results.append({
                    'Dataset': dataset_name.upper(),
                    'Feature_Set': feat_name,
                    'Model': model_name,
                    'Accuracy': acc,
                    'F1_Macro': f1
                })
                print(f"   {feat_name:<35} + {model_name:<20} | Acc: {acc:.4f} | F1: {f1:.4f}")
            except Exception as e:
                print(f"   {feat_name} + {model_name} failed: {str(e)[:50]}...")
        gc.collect()
        
    res_df = pd.DataFrame(results)
    best_f1 = res_df['F1_Macro'].max()
    improvement = best_f1 - feb_baseline_f1
    
    print(f"\n BEST F1 for {dataset_name.upper()}: {best_f1:.4f}")
    print(f" Improvement over Feb Baseline ({feb_baseline_f1:.4f}): {improvement:+.4f}")
    return res_df


duo_res = run_collatex_enhanced_experiment('clean_duo_data.csv', 'duo', feb_baseline_f1=0.882)
trio_res = run_collatex_enhanced_experiment('clean_trio_data.csv', 'trio', feb_baseline_f1=0.788)

full_results = pd.concat([duo_res, trio_res], ignore_index=True)

print("\n" + "="*70)
print(" TOP 10 RESULTS ACROSS BOTH DATASETS (Sorted by F1)")
print("="*70)
print(full_results.sort_values('F1_Macro', ascending=False).head(10).to_string(index=False))




📊 FULL EXPERIMENT GRID - DUO
📝 Extracting features...
🚀 Running experiments...

  ✅ TF-IDF Word (1,2)              + Logistic Regression  | Acc: 0.7969 | F1: 0.7781
  ✅ TF-IDF Word (1,2)              + Naive Bayes          | Acc: 0.7406 | F1: 0.7141
  ✅ TF-IDF Word (1,2)              + SVM                  | Acc: 0.7995 | F1: 0.7820
  ✅ TF-IDF Word (1,2)              + KNN                  | Acc: 0.5864 | F1: 0.5825
  ✅ TF-IDF Word (1,2)              + SGD                  | Acc: 0.7784 | F1: 0.7574
  ✅ TF-IDF Char (2,4)              + Logistic Regression  | Acc: 0.8419 | F1: 0.8287
  ✅ TF-IDF Char (2,4)              + Naive Bayes          | Acc: 0.8207 | F1: 0.8100
  ✅ TF-IDF Char (2,4)              + SVM                  | Acc: 0.8191 | F1: 0.8024
  ✅ TF-IDF Char (2,4)              + KNN                  | Acc: 0.7602 | F1: 0.7368
  ✅ TF-IDF Char (2,4)              + SGD                  | Acc: 0.8187 | F1: 0.8036
  ✅ Word2Vec (100D)                + Logistic Regression  | Acc: 0.71